In [1]:
import os
import json
import pickle
import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report

from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ResNet-50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

2026-10-27 04:59:25.759952: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793077165.994346      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793077166.059656      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793077166.629445      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793077166.629490      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793077166.629494      58 computation_placer.cc:177] computation placer alr

In [2]:
SEEDS = [42, 123, 2024, 3407, 777]

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 5
LEARNING_RATE = 1e-5
FINE_TUNE_LAYERS = 20
AUTOTUNE = tf.data.AUTOTUNE

CANONICAL_CLASSES = ["Early Blight", "Late Blight", "Healthy"]
NUM_CLASSES = len(CANONICAL_CLASSES)

SOURCE_DATASET_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DATASET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"

RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = f"cross_dataset_transfer_{RUN_TIMESTAMP}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "histories"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "reports"), exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: cross_dataset_transfer_20261027-050517


In [3]:
CLASS_FOLDER_CANDIDATES = {
    "Early Blight": [
        "Early Blight",
        "Early_Blight",
        "Potato___Early_blight",
        "Potato___Early_Blight"
    ],
    "Late Blight": [
        "Late Blight",
        "Late_Blight",
        "Potato___Late_blight",
        "Potato___Late_Blight"
    ],
    "Healthy": [
        "Healthy",
        "healthy",
        "Potato___healthy",
        "Potato___Healthy"
    ]
}


def find_class_folder(dataset_dir, canonical_class):
    available_folders = [
        folder for folder in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, folder))
    ]

    for candidate in CLASS_FOLDER_CANDIDATES[canonical_class]:
        if candidate in available_folders:
            return os.path.join(dataset_dir, candidate)

    raise ValueError(
        f"Could not find folder for class '{canonical_class}' in {dataset_dir}. "
        f"Available folders: {available_folders}"
    )


def collect_paths_and_labels(dataset_dir):
    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    file_paths = []
    labels = []

    for label_index, class_name in enumerate(CANONICAL_CLASSES):
        class_dir = find_class_folder(dataset_dir, class_name)

        for fname in os.listdir(class_dir):
            if fname.lower().endswith(valid_extensions):
                file_paths.append(os.path.join(class_dir, fname))
                labels.append(label_index)

    file_paths = np.array(file_paths)
    labels = np.array(labels)

    print("\nDataset:", dataset_dir)
    print("Total images:", len(labels))

    unique, counts = np.unique(labels, return_counts=True)
    for idx, count in zip(unique, counts):
        print(f"{CANONICAL_CLASSES[idx]}: {count}")

    return file_paths, labels

In [4]:
source_paths, source_labels = collect_paths_and_labels(SOURCE_DATASET_DIR)
target_paths, target_labels = collect_paths_and_labels(TARGET_DATASET_DIR)


Dataset: /kaggle/input/datasets/faysalmiah1721758/potato-dataset
Total images: 2152
Early Blight: 1000
Late Blight: 1000
Healthy: 152

Dataset: /kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease
Total images: 1500
Early Blight: 500
Late Blight: 500
Healthy: 500


In [5]:
def make_splits(file_paths, labels, seed):
    X_train, X_temp, y_train, y_temp = train_test_split(
        file_paths,
        labels,
        test_size=0.20,
        stratify=labels,
        random_state=seed
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=seed
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [6]:
def load_tf_dataset(file_paths, labels, image_size, preprocess_fn, augment=False, shuffle=True, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))

    def process(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])

        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32)

        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, max_delta=25.5)
            image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
            image = tf.clip_by_value(image, 0.0, 255.0)

        image = preprocess_fn(image)
        label = tf.one_hot(label, depth=NUM_CLASSES)

        return image, label

    ds = ds.map(process, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(file_paths),
            seed=seed,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds

In [7]:
MODEL_CONFIGS = {
    "ResNet50": {
        "image_size": (224, 224),
        "preprocess_fn": resnet_preprocess
    }
}

In [8]:
def build_resnet50(input_shape=(224, 224, 3), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)

    backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    # Fine-tune the last 20 layers
    backbone.trainable = True

    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False

    # Keep BatchNormalization layers frozen
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="ResNet50_transfer"
    )

    return model


MODEL_BUILDERS = {
    "ResNet50": build_resnet50
}

In [9]:
def get_class_weights(y_train):
    class_weight_vals = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=y_train
    )

    return dict(enumerate(class_weight_vals))

In [10]:
def run_cross_dataset_experiment(model_name, seed):
    print("\n" + "=" * 80)
    print(f"Model: {model_name} | Seed: {seed}")
    print("=" * 80)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    image_size = MODEL_CONFIGS[model_name]["image_size"]
    preprocess_fn = MODEL_CONFIGS[model_name]["preprocess_fn"]

    source_train, source_val, source_test, y_source_train, y_source_val, y_source_test = make_splits(
        source_paths,
        source_labels,
        seed
    )

    target_train, target_val, target_test, y_target_train, y_target_val, y_target_test = make_splits(
        target_paths,
        target_labels,
        seed
    )

    source_train_ds = load_tf_dataset(source_train, y_source_train, image_size, preprocess_fn, augment=True, shuffle=True, seed=seed)
    source_val_ds = load_tf_dataset(source_val, y_source_val, image_size, preprocess_fn, augment=False, shuffle=False)
    source_test_ds = load_tf_dataset(source_test, y_source_test, image_size, preprocess_fn, augment=False, shuffle=False)

    target_train_ds = load_tf_dataset(target_train, y_target_train, image_size, preprocess_fn, augment=True, shuffle=True, seed=seed)
    target_val_ds = load_tf_dataset(target_val, y_target_val, image_size, preprocess_fn, augment=False, shuffle=False)
    target_test_ds = load_tf_dataset(target_test, y_target_test, image_size, preprocess_fn, augment=False, shuffle=False)

    model = MODEL_BUILDERS[model_name](
        input_shape=(image_size[0], image_size[1], 3),
        num_classes=NUM_CLASSES
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    safe_model_name = model_name.replace(" ", "_")
    run_name = f"{safe_model_name}_seed_{seed}"

    source_model_path = os.path.join(OUTPUT_DIR, "models", f"{run_name}_source.keras")
    finetuned_model_path = os.path.join(OUTPUT_DIR, "models", f"{run_name}_target_finetuned.keras")

    source_history_path = os.path.join(OUTPUT_DIR, "histories", f"{run_name}_source_history.pkl")
    target_history_path = os.path.join(OUTPUT_DIR, "histories", f"{run_name}_target_finetune_history.pkl")

    source_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            source_model_path,
            monitor="val_loss",
            save_best_only=True
        )
    ]

    source_history = model.fit(
        source_train_ds,
        validation_data=source_val_ds,
        epochs=EPOCHS,
        callbacks=source_callbacks,
        class_weight=get_class_weights(y_source_train),
        verbose=1
    )

    with open(source_history_path, "wb") as f:
        pickle.dump(source_history.history, f)

    source_eval = evaluate_model(model, source_test_ds)
    zero_shot_eval = evaluate_model(model, target_test_ds)

    target_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            finetuned_model_path,
            monitor="val_loss",
            save_best_only=True
        )
    ]

    target_history = model.fit(
        target_train_ds,
        validation_data=target_val_ds,
        epochs=EPOCHS,
        callbacks=target_callbacks,
        class_weight=get_class_weights(y_target_train),
        verbose=1
    )

    with open(target_history_path, "wb") as f:
        pickle.dump(target_history.history, f)

    finetuned_eval = evaluate_model(model, target_test_ds)

    source_acc = source_eval["accuracy"]
    zero_acc = zero_shot_eval["accuracy"]
    fine_acc = finetuned_eval["accuracy"]

    result = {
        "model": model_name,
        "seed": seed,

        "source_loss": source_eval["loss"],
        "source_accuracy": source_acc,
        "source_macro_f1": source_eval["macro_f1"],
        "source_weighted_f1": source_eval["weighted_f1"],

        "zero_shot_loss": zero_shot_eval["loss"],
        "zero_shot_accuracy": zero_acc,
        "zero_shot_macro_f1": zero_shot_eval["macro_f1"],
        "zero_shot_weighted_f1": zero_shot_eval["weighted_f1"],

        "finetuned_loss": finetuned_eval["loss"],
        "finetuned_accuracy": fine_acc,
        "finetuned_macro_f1": finetuned_eval["macro_f1"],
        "finetuned_weighted_f1": finetuned_eval["weighted_f1"],

        "zero_shot_drop": source_acc - zero_acc,
        "finetuned_drop": source_acc - fine_acc,
        "zero_shot_ratio": zero_acc / source_acc if source_acc > 0 else np.nan,
        "finetuned_ratio": fine_acc / source_acc if source_acc > 0 else np.nan,

        "source_model_path": source_model_path,
        "finetuned_model_path": finetuned_model_path,
        "source_history_path": source_history_path,
        "target_history_path": target_history_path
    }

    print("\nResult:")
    print(result)

    return result

In [11]:
from sklearn.metrics import f1_score

def evaluate_model(model, dataset):
    y_true = []
    y_pred = []
    y_prob = []

    # Evaluate model
    loss, acc = model.evaluate(dataset, verbose=0)

    # Generate predictions
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)

        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(probs, axis=1))
        y_prob.extend(probs)

    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)
    y_prob = np.asarray(y_prob, dtype=np.float32)

    macro_f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": float(loss),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

In [12]:
all_transfer_results = []

model_name = "ResNet50"

for seed in SEEDS:
    print(f"\n{'=' * 80}")
    print(f"Running {model_name} | Seed: {seed}")
    print(f"{'=' * 80}")

    result = run_cross_dataset_experiment(model_name, seed)

    all_transfer_results.append(result)

    # Save after every completed seed
    per_seed_df = pd.DataFrame(all_transfer_results)

    per_seed_csv = os.path.join(
        OUTPUT_DIR,
        "resnet50_cross_dataset_transfer_per_seed_results.csv"
    )

    per_seed_df.to_csv(per_seed_csv, index=False)

    print(f"\nSeed {seed} completed.")
    print(f"Results saved to: {per_seed_csv}")

print("\nAll ResNet-50 cross-dataset experiments completed.")

per_seed_df


Running ResNet50 | Seed: 42

Model: ResNet50 | Seed: 42


I0000 00:00:1793078682.379207      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793078682.385370      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50


I0000 00:00:1793078764.028129     136 service.cc:152] XLA service 0x7d7ca0115990 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793078764.028174     136 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1793078764.028181     136 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1793078766.108143     136 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/54 ━━━━━━━━━━━━━━━━━━━━ 1:16:45 87s/step - accuracy: 0.3750 - loss: 1.0922

I0000 00:00:1793078774.089158     136 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


54/54 ━━━━━━━━━━━━━━━━━━━━ 117s 570ms/step - accuracy: 0.6891 - loss: 0.8296 - val_accuracy: 0.9488 - val_loss: 0.1532
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 10s 130ms/step - accuracy: 0.9471 - loss: 0.1364 - val_accuracy: 0.9721 - val_loss: 0.0905
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 10s 133ms/step - accuracy: 0.9814 - loss: 0.0512 - val_accuracy: 0.9721 - val_loss: 0.0516
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 114ms/step - accuracy: 0.9895 - loss: 0.0399 - val_accuracy: 0.9814 - val_loss: 0.0616
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 117ms/step - accuracy: 0.9901 - loss: 0.0170 - val_accuracy: 0.9767 - val_loss: 0.0579
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 10s 139ms/step - accuracy: 0.9965 - loss: 0.0130 - val_accuracy: 0.9860 - val_loss: 0.0406
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 121ms/step - accuracy: 0.9965 - loss: 0.0091 - val_accuracy: 0.9767 - val_loss: 0.0539
Epoch 8/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 123ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 0.9767 -

,model,seed,source_loss,source_accuracy,source_macro_f1,source_weighted_f1,zero_shot_loss,zero_shot_accuracy,zero_shot_macro_f1,zero_shot_weighted_f1,...,finetuned_macro_f1,finetuned_weighted_f1,zero_shot_drop,finetuned_drop,zero_shot_ratio,finetuned_ratio,source_model_path,finetuned_model_path,source_history_path,target_history_path
0,ResNet50,42,0.052427,0.976852,0.965780,0.976778,0.592136,0.880000,0.876652,0.876652,...,0.967015,0.967015,0.096852,0.010185,0.900853,0.989573,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/histori...,cross_dataset_transfer_20261027-050517/histori...
1,ResNet50,123,0.076239,0.976852,0.945670,0.976394,0.673034,0.866667,0.862070,0.862070,...,0.979998,0.979998,0.110185,-0.003148,0.887204,1.003223,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/histori...,cross_dataset_transfer_20261027-050517/histori...
2,ResNet50,2024,0.010456,0.990741,0.984874,0.990776,0.773722,0.906667,0.904630,0.904630,...,0.986661,0.986661,0.084074,0.004074,0.915140,0.995888,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/histori...,cross_dataset_transfer_20261027-050517/histori...
3,ResNet50,3407,0.054077,0.986111,0.962767,0.985922,1.072399,0.840000,0.831559,0.831559,...,0.993333,0.993333,0.146111,-0.007222,0.851831,1.007324,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/histori...,cross_dataset_transfer_20261027-050517/histori...
4,ResNet50,777,0.091919,0.986111,0.964672,0.986287,0.857248,0.880000,0.877305,0.877305,...,0.959557,0.959557,0.106111,0.026111,0.892394,0.973521,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/models/...,cross_dataset_transfer_20261027-050517/histori...,cross_dataset_transfer_20261027-050517/histori...


In [13]:
def mean_std_percent(series):
    mean = series.mean() * 100
    std = series.std(ddof=0) * 100
    return f"{mean:.2f} ± {std:.2f}"


def mean_std_ratio(series):
    mean = series.mean()
    std = series.std(ddof=0)
    return f"{mean:.4f} ± {std:.4f}"


summary_rows = []

for model_name, group in per_seed_df.groupby("model"):
    summary_rows.append({
        "Model": model_name,
        "Source Acc (%)": mean_std_percent(group["source_accuracy"]),
        "Zero-shot Acc (%)": mean_std_percent(group["zero_shot_accuracy"]),
        "Fine-tuned Acc (%)": mean_std_percent(group["finetuned_accuracy"]),
        "Zero-shot Drop (%)": mean_std_percent(group["zero_shot_drop"]),
        "Fine-tuned Drop (%)": mean_std_percent(group["finetuned_drop"]),
        "Zero-shot Ratio": mean_std_ratio(group["zero_shot_ratio"]),
        "Fine-tuned Ratio": mean_std_ratio(group["finetuned_ratio"]),
        "Source Macro F1": mean_std_ratio(group["source_macro_f1"]),
        "Zero-shot Macro F1": mean_std_ratio(group["zero_shot_macro_f1"]),
        "Fine-tuned Macro F1": mean_std_ratio(group["finetuned_macro_f1"]),
        "Source Weighted F1": mean_std_ratio(group["source_weighted_f1"]),
        "Zero-shot Weighted F1": mean_std_ratio(group["zero_shot_weighted_f1"]),
        "Fine-tuned Weighted F1": mean_std_ratio(group["finetuned_weighted_f1"])
    })

summary_df = pd.DataFrame(summary_rows)

os.makedirs(OUTPUT_DIR, exist_ok=True)

summary_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_cross_dataset_transfer_results.csv"
)

summary_df.to_csv(summary_path, index=False)

print("Saved summary results to:", summary_path)

summary_df

Saved summary results to: cross_dataset_transfer_20261027-050517/resnet50_cross_dataset_transfer_results.csv


,Model,Source Acc (%),Zero-shot Acc (%),Fine-tuned Acc (%),Zero-shot Drop (%),Fine-tuned Drop (%),Zero-shot Ratio,Fine-tuned Ratio,Source Macro F1,Zero-shot Macro F1,Fine-tuned Macro F1,Source Weighted F1,Zero-shot Weighted F1,Fine-tuned Weighted F1
0,ResNet50,98.33 ± 0.56,87.47 ± 2.17,97.73 ± 1.24,10.87 ± 2.08,0.60 ± 1.17,0.8895 ± 0.0211,0.9939 ± 0.0119,0.9648 ± 0.0124,0.8704 ± 0.0238,0.9773 ± 0.0124,0.9832 ± 0.0057,0.8704 ± 0.0238,0.9773 ± 0.0124


In [14]:
paper_table = summary_df[
    [
        "Model",
        "Source Acc (%)",
        "Zero-shot Acc (%)",
        "Fine-tuned Acc (%)",
        "Zero-shot Drop (%)",
        "Fine-tuned Drop (%)",
        "Zero-shot Ratio",
        "Fine-tuned Ratio"
    ]
].copy()

paper_table_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_cross_dataset_transfer_paper_table.csv"
)

paper_table.to_csv(paper_table_path, index=False)

print("Saved paper table to:", paper_table_path)

paper_table

Saved paper table to: cross_dataset_transfer_20261027-050517/resnet50_cross_dataset_transfer_paper_table.csv


,Model,Source Acc (%),Zero-shot Acc (%),Fine-tuned Acc (%),Zero-shot Drop (%),Fine-tuned Drop (%),Zero-shot Ratio,Fine-tuned Ratio
0,ResNet50,98.33 ± 0.56,87.47 ± 2.17,97.73 ± 1.24,10.87 ± 2.08,0.60 ± 1.17,0.8895 ± 0.0211,0.9939 ± 0.0119


In [15]:
zip_name = f"{OUTPUT_DIR}.zip"

!zip -r {zip_name} {OUTPUT_DIR}

print("Created:", zip_name)

  adding: cross_dataset_transfer_20261027-050517/ (stored 0%)
  adding: cross_dataset_transfer_20261027-050517/models/ (stored 0%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_3407_source.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_42_target_finetuned.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_3407_target_finetuned.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_2024_target_finetuned.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_777_target_finetuned.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_123_source.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_2024_source.keras (deflated 7%)
  adding: cross_dataset_transfer_20261027-050517/models/ResNet50_seed_42_source.keras (deflated 7%)
  adding: cross_dat

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ResNet-50 cross-dataset transfer prediction files
# These will be populated after predictions are saved.

PREDICTION_FILES = {}

In [17]:
import os

print("Available CSV files:")

csv_files = []

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.lower().endswith(".csv"):
            file_path = os.path.join(root, file)
            csv_files.append(file_path)

for file_path in sorted(csv_files):
    print(file_path)

print(f"\nTotal CSV files found: {len(csv_files)}")

Available CSV files:
/kaggle/working/cross_dataset_transfer_20261027-050517/resnet50_cross_dataset_transfer_paper_table.csv
/kaggle/working/cross_dataset_transfer_20261027-050517/resnet50_cross_dataset_transfer_per_seed_results.csv
/kaggle/working/cross_dataset_transfer_20261027-050517/resnet50_cross_dataset_transfer_results.csv

Total CSV files found: 3


In [18]:
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.model_selection import train_test_split

CANONICAL_CLASSES = ["Early Blight", "Late Blight", "Healthy"]
NUM_CLASSES = 3

SEEDS = [42, 123, 2024, 3407, 777]

SOURCE_DATASET_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DATASET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"

MODEL_SETTINGS = {
    "ResNet50": {
        "display_name": "ResNet50",
        "image_size": (224, 224),
        "preprocess_fn": resnet_preprocess
    }
}

def load_image_for_prediction(image_path, image_size, preprocess_fn):
    image_raw = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image_raw,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32)

    image = preprocess_fn(image)
    image = tf.expand_dims(image, axis=0)

    return image


def generate_prediction_csv(
    model_path,
    file_paths,
    labels,
    model_key,
    dataset_name,
    seed,
    output_dir
):
    model = load_model(model_path, compile=False)

    image_size = MODEL_SETTINGS[model_key]["image_size"]
    preprocess_fn = MODEL_SETTINGS[model_key]["preprocess_fn"]

    records = []

    for image_path, true_label in zip(file_paths, labels):
        image = load_image_for_prediction(
            image_path,
            image_size,
            preprocess_fn
        )

        probs = model.predict(image, verbose=0)[0]

        pred_label = int(np.argmax(probs))
        confidence = float(np.max(probs))

        record = {
            "image_path": image_path,
            "true_label": int(true_label),
            "true_class": CANONICAL_CLASSES[int(true_label)],
            "predicted_label": pred_label,
            "pred_class": CANONICAL_CLASSES[pred_label],
            "confidence": confidence,
            "correct": int(true_label) == pred_label,
            "model": MODEL_SETTINGS[model_key]["display_name"],
            "dataset": dataset_name,
            "seed": seed
        }

        for class_index, class_name in enumerate(CANONICAL_CLASSES):
            record[f"prob_class_{class_index}"] = float(probs[class_index])

        records.append(record)

    pred_df = pd.DataFrame(records)

    os.makedirs(output_dir, exist_ok=True)

    csv_path = os.path.join(
        output_dir,
        f"test_predictions_{model_key}_{dataset_name}_seed_{seed}.csv"
    )

    pred_df.to_csv(csv_path, index=False)

    print("Saved:", csv_path)

    return csv_path

In [19]:
source_paths, source_labels = collect_paths_and_labels(SOURCE_DATASET_DIR)
target_paths, target_labels = collect_paths_and_labels(TARGET_DATASET_DIR)

# Use the current ResNet-50 experiment directory
BASE_DIR = OUTPUT_DIR

prediction_output_dir = os.path.join(BASE_DIR, "prediction_csvs")
os.makedirs(prediction_output_dir, exist_ok=True)

generated_prediction_files = []

model_key = "ResNet50"

for seed in SEEDS:
    print(f"\nGenerating predictions for {model_key} | Seed {seed}")

    _, _, source_test, _, _, y_source_test = make_splits(
        source_paths,
        source_labels,
        seed
    )

    _, _, target_test, _, _, y_target_test = make_splits(
        target_paths,
        target_labels,
        seed
    )

    source_model_path = os.path.join(
        BASE_DIR,
        "models",
        f"{model_key}_seed_{seed}_source.keras"
    )

    target_model_path = os.path.join(
        BASE_DIR,
        "models",
        f"{model_key}_seed_{seed}_target_finetuned.keras"
    )

    # Source-domain and zero-shot evaluations
    if os.path.exists(source_model_path):
        generated_prediction_files.append(
            generate_prediction_csv(
                model_path=source_model_path,
                file_paths=source_test,
                labels=y_source_test,
                model_key=model_key,
                dataset_name="source_3class",
                seed=seed,
                output_dir=prediction_output_dir
            )
        )

        generated_prediction_files.append(
            generate_prediction_csv(
                model_path=source_model_path,
                file_paths=target_test,
                labels=y_target_test,
                model_key=model_key,
                dataset_name="zero_shot_target_3class",
                seed=seed,
                output_dir=prediction_output_dir
            )
        )
    else:
        print(f"Missing source checkpoint: {source_model_path}")

    # Fine-tuned target-domain evaluation
    if os.path.exists(target_model_path):
        generated_prediction_files.append(
            generate_prediction_csv(
                model_path=target_model_path,
                file_paths=target_test,
                labels=y_target_test,
                model_key=model_key,
                dataset_name="finetuned_target_3class",
                seed=seed,
                output_dir=prediction_output_dir
            )
        )
    else:
        print(f"Missing fine-tuned checkpoint: {target_model_path}")

print(f"\nGenerated {len(generated_prediction_files)} prediction CSV files.")

for path in generated_prediction_files:
    print(path)


Dataset: /kaggle/input/datasets/faysalmiah1721758/potato-dataset
Total images: 2152
Early Blight: 1000
Late Blight: 1000
Healthy: 152

Dataset: /kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease
Total images: 1500
Early Blight: 500
Late Blight: 500
Healthy: 500

Generating predictions for ResNet50 | Seed 42
Saved: cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_source_3class_seed_42.csv
Saved: cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_zero_shot_target_3class_seed_42.csv
Saved: cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_finetuned_target_3class_seed_42.csv

Generating predictions for ResNet50 | Seed 123
Saved: cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_source_3class_seed_123.csv
Saved: cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_zero_shot_target_3class_seed_123.csv

In [20]:
# Locate ResNet-50 prediction CSV files
prediction_dir = os.path.join(OUTPUT_DIR, "prediction_csvs")

prediction_files = []

for root, dirs, files in os.walk(prediction_dir):
    for file in files:
        if (
            file.endswith(".csv")
            and file.startswith("test_predictions_ResNet50_")
        ):
            prediction_files.append(os.path.join(root, file))

prediction_files = sorted(prediction_files)

print("Found ResNet-50 prediction files:", len(prediction_files))

for path in prediction_files:
    print(path)


# Validate and load prediction files
required_columns = [
    "image_path",
    "true_class",
    "pred_class",
    "confidence",
    "correct",
    "model",
    "dataset",
    "seed"
]

prediction_dfs = []

for csv_path in prediction_files:
    df = pd.read_csv(csv_path)

    missing_cols = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"{csv_path} missing columns: {missing_cols}"
        )

    df["confidence"] = df["confidence"].astype(float)

    if df["correct"].dtype != bool:
        df["correct"] = (
            df["correct"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
        )

    prediction_dfs.append(df)


# Combine predictions
if not prediction_dfs:
    raise FileNotFoundError(
        f"No ResNet-50 prediction CSV files found in {prediction_dir}. "
        "Complete training and prediction export first."
    )

all_predictions_df = pd.concat(
    prediction_dfs,
    ignore_index=True
)

print("\nTotal prediction rows:", len(all_predictions_df))

print("\nPrediction files by dataset:")
print(
    all_predictions_df.groupby(["model", "dataset"])["seed"]
    .nunique()
)

display(all_predictions_df.head())

Found ResNet-50 prediction files: 15
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_finetuned_target_3class_seed_123.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_finetuned_target_3class_seed_2024.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_finetuned_target_3class_seed_3407.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_finetuned_target_3class_seed_42.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_finetuned_target_3class_seed_777.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_source_3class_seed_123.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_source_3class_seed_2024.csv
cross_dataset_transfer_20261027-050517/prediction_csvs/test_predictions_ResNet50_source_3class_seed_3407.csv
cross_dataset_transfer_20261027-050517/predict

,image_path,true_label,true_class,predicted_label,pred_class,confidence,correct,model,dataset,seed,prob_class_0,prob_class_1,prob_class_2
0,/kaggle/input/datasets/shahadhossin567r7455/po...,1,Late Blight,1,Late Blight,1.000000,True,ResNet50,finetuned_target_3class,123,2.074526e-07,1.000000,1.880786e-10
1,/kaggle/input/datasets/shahadhossin567r7455/po...,0,Early Blight,0,Early Blight,0.999968,True,ResNet50,finetuned_target_3class,123,9.999679e-01,0.000028,4.038684e-06
2,/kaggle/input/datasets/shahadhossin567r7455/po...,2,Healthy,2,Healthy,0.982462,True,ResNet50,finetuned_target_3class,123,7.203928e-05,0.017466,9.824616e-01
3,/kaggle/input/datasets/shahadhossin567r7455/po...,2,Healthy,2,Healthy,0.998928,True,ResNet50,finetuned_target_3class,123,8.294894e-05,0.000989,9.989279e-01
4,/kaggle/input/datasets/shahadhossin567r7455/po...,2,Healthy,2,Healthy,0.969049,True,ResNet50,finetuned_target_3class,123,4.816536e-05,0.030903,9.690492e-01


In [21]:
def compute_decision_metrics(df, scheme_name, alpha=None, beta=None):
    total_samples = len(df)
    total_wrong = (~df["correct"]).sum()

    accept_df = df[df["decision_region"] == "Accept"]
    defer_df = df[df["decision_region"] == "Defer"]
    reject_df = df[df["decision_region"] == "Reject"]

    accept_count = len(accept_df)
    defer_count = len(defer_df)
    reject_count = len(reject_df)

    accept_percentage = accept_count / total_samples * 100
    defer_percentage = defer_count / total_samples * 100
    reject_percentage = reject_count / total_samples * 100

    if accept_count > 0:
        accept_accuracy = accept_df["correct"].mean() * 100
    else:
        accept_accuracy = np.nan

    captured_wrong = df[
        (~df["correct"]) &
        (df["decision_region"].isin(["Defer", "Reject"]))
    ]

    if total_wrong > 0:
        wrong_capture_rate = len(captured_wrong) / total_wrong * 100
    else:
        wrong_capture_rate = np.nan

    return {
        "scheme": scheme_name,
        "alpha": alpha,
        "beta": beta,
        "total_samples": total_samples,
        "accept_count": accept_count,
        "defer_count": defer_count,
        "reject_count": reject_count,
        "accept_percentage": accept_percentage,
        "defer_percentage": defer_percentage,
        "reject_percentage": reject_percentage,
        "accept_accuracy": accept_accuracy,
        "wrong_capture_rate": wrong_capture_rate,
        "total_wrong": int(total_wrong),
        "captured_wrong": int(len(captured_wrong))
    }

In [22]:
two_way_alphas = [0.85, 0.90, 0.95]

three_way_thresholds = [
    (0.95, 0.70),
    (0.90, 0.60),
    (0.85, 0.50)
]

ablation_rows = []

group_cols = ["model", "dataset", "seed"]

for (model_name, dataset_name, seed), group_df in all_predictions_df.groupby(group_cols):
    group_df = group_df.copy()

    # A. Standard hard prediction
    standard_df = group_df.copy()
    standard_df["decision_region"] = "Accept"

    result = compute_decision_metrics(
        standard_df,
        scheme_name="Standard",
        alpha=None,
        beta=None
    )
    result["model"] = model_name
    result["dataset"] = dataset_name
    result["seed"] = seed
    ablation_rows.append(result)

    # B. Two-way decision
    for alpha in two_way_alphas:
        two_way_df = group_df.copy()
        two_way_df["decision_region"] = np.where(
            two_way_df["confidence"] >= alpha,
            "Accept",
            "Reject"
        )

        result = compute_decision_metrics(
            two_way_df,
            scheme_name="Two-way",
            alpha=alpha,
            beta=None
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        ablation_rows.append(result)

    # C. Three-way decision
    for alpha, beta in three_way_thresholds:
        three_way_df = group_df.copy()

        conditions = [
            three_way_df["confidence"] >= alpha,
            (three_way_df["confidence"] >= beta) & (three_way_df["confidence"] < alpha),
            three_way_df["confidence"] < beta
        ]

        choices = ["Accept", "Defer", "Reject"]

        three_way_df["decision_region"] = np.select(
            conditions,
            choices,
            default="Reject"
        )

        result = compute_decision_metrics(
            three_way_df,
            scheme_name="Three-way",
            alpha=alpha,
            beta=beta
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        ablation_rows.append(result)

ablation_df = pd.DataFrame(ablation_rows)

ablation_df = ablation_df[
    [
        "model",
        "dataset",
        "seed",
        "scheme",
        "alpha",
        "beta",
        "total_samples",
        "accept_count",
        "defer_count",
        "reject_count",
        "accept_percentage",
        "defer_percentage",
        "reject_percentage",
        "accept_accuracy",
        "wrong_capture_rate",
        "total_wrong",
        "captured_wrong"
    ]
]

ablation_results_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_ablation_decision_mechanism_results.csv"
)

ablation_df.to_csv(ablation_results_path, index=False)

print("Saved:", ablation_results_path)
display(ablation_df.head())

Saved: cross_dataset_transfer_20261027-050517/resnet50_ablation_decision_mechanism_results.csv


,model,dataset,seed,scheme,alpha,beta,total_samples,accept_count,defer_count,reject_count,accept_percentage,defer_percentage,reject_percentage,accept_accuracy,wrong_capture_rate,total_wrong,captured_wrong
0,ResNet50,finetuned_target_3class,42,Standard,NaN,NaN,150,150,0,0,100.000000,0.0,0.000000,96.666667,0.0,5,0
1,ResNet50,finetuned_target_3class,42,Two-way,0.85,NaN,150,139,0,11,92.666667,0.0,7.333333,99.280576,80.0,5,4
2,ResNet50,finetuned_target_3class,42,Two-way,0.90,NaN,150,138,0,12,92.000000,0.0,8.000000,100.000000,100.0,5,5
3,ResNet50,finetuned_target_3class,42,Two-way,0.95,NaN,150,136,0,14,90.666667,0.0,9.333333,100.000000,100.0,5,5
4,ResNet50,finetuned_target_3class,42,Three-way,0.95,0.7,150,136,6,8,90.666667,4.0,5.333333,100.000000,100.0,5,5


In [23]:
summary_metrics = [
    "accept_percentage",
    "defer_percentage",
    "reject_percentage",
    "accept_accuracy",
    "wrong_capture_rate"
]

summary_rows = []

for keys, group in ablation_df.groupby(
    ["model", "dataset", "scheme", "alpha", "beta"],
    dropna=False
):
    model_name, dataset_name, scheme, alpha, beta = keys

    row = {
        "model": model_name,
        "dataset": dataset_name,
        "scheme": scheme,
        "alpha": alpha,
        "beta": beta
    }

    for metric in summary_metrics:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=0)

    summary_rows.append(row)

ablation_summary_df = pd.DataFrame(summary_rows)

ablation_summary_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_ablation_decision_mechanism_summary_mean_std.csv"
)

ablation_summary_df.to_csv(
    ablation_summary_path,
    index=False
)

print("Saved:", ablation_summary_path)

display(ablation_summary_df.head())

Saved: cross_dataset_transfer_20261027-050517/resnet50_ablation_decision_mechanism_summary_mean_std.csv


,model,dataset,scheme,alpha,beta,accept_percentage_mean,accept_percentage_std,defer_percentage_mean,defer_percentage_std,reject_percentage_mean,reject_percentage_std,accept_accuracy_mean,accept_accuracy_std,wrong_capture_rate_mean,wrong_capture_rate_std
0,ResNet50,finetuned_target_3class,Standard,NaN,NaN,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,97.733333,1.236482,0.000000,0.000000
1,ResNet50,finetuned_target_3class,Three-way,0.85,0.5,96.666667,2.422120,3.333333,2.422120,0.000000,0.000000,99.173826,0.504385,62.666667,33.625387
2,ResNet50,finetuned_target_3class,Three-way,0.90,0.6,96.266667,2.686592,2.800000,2.362673,0.933333,0.997775,99.452768,0.505166,70.000000,37.118429
3,ResNet50,finetuned_target_3class,Three-way,0.95,0.7,95.733333,2.969474,2.133333,0.979796,2.133333,2.039608,99.724020,0.338126,83.333333,21.081851
4,ResNet50,finetuned_target_3class,Two-way,0.85,NaN,96.666667,2.422120,0.000000,0.000000,3.333333,2.422120,99.173826,0.504385,62.666667,33.625387


In [24]:
table_a = ablation_summary_df[
    (
        (ablation_summary_df["scheme"] == "Standard") |
        (
            (ablation_summary_df["scheme"] == "Two-way") &
            (ablation_summary_df["alpha"] == 0.90)
        ) |
        (
            (ablation_summary_df["scheme"] == "Three-way") &
            (ablation_summary_df["alpha"] == 0.90) &
            (ablation_summary_df["beta"] == 0.60)
        )
    )
].copy()

def format_mean_std(row, metric):
    return f"{row[f'{metric}_mean']:.2f} ± {row[f'{metric}_std']:.2f}"

table_a["setting"] = table_a.apply(
    lambda row: (
        "Standard"
        if row["scheme"] == "Standard"
        else "Two-way α=0.90"
        if row["scheme"] == "Two-way"
        else "Three-way α=0.90, β=0.60"
    ),
    axis=1
)

table_a_formatted = pd.DataFrame({
    "Model": table_a["model"],
    "Dataset": table_a["dataset"],
    "Setting": table_a["setting"],
    "Accept (%)": table_a.apply(lambda row: format_mean_std(row, "accept_percentage"), axis=1),
    "Defer (%)": table_a.apply(lambda row: format_mean_std(row, "defer_percentage"), axis=1),
    "Reject (%)": table_a.apply(lambda row: format_mean_std(row, "reject_percentage"), axis=1),
    "Accept Accuracy (%)": table_a.apply(lambda row: format_mean_std(row, "accept_accuracy"), axis=1),
    "Wrong Capture Rate (%)": table_a.apply(lambda row: format_mean_std(row, "wrong_capture_rate"), axis=1)
})

table_a_formatted = table_a_formatted.sort_values(["Dataset", "Model", "Setting"])

table_a_path = "ablation_table_A_standard_two_way_three_way.csv"
table_a_formatted.to_csv(table_a_path, index=False)

print("Saved:", table_a_path)
display(table_a_formatted)

Saved: ablation_table_A_standard_two_way_three_way.csv


,Model,Dataset,Setting,Accept (%),Defer (%),Reject (%),Accept Accuracy (%),Wrong Capture Rate (%)
0,ResNet50,finetuned_target_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,97.73 ± 1.24,0.00 ± 0.00
2,ResNet50,finetuned_target_3class,"Three-way α=0.90, β=0.60",96.27 ± 2.69,2.80 ± 2.36,0.93 ± 1.00,99.45 ± 0.51,70.00 ± 37.12
5,ResNet50,finetuned_target_3class,Two-way α=0.90,96.27 ± 2.69,0.00 ± 0.00,3.73 ± 2.69,99.45 ± 0.51,70.00 ± 37.12
7,ResNet50,source_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,98.33 ± 0.56,0.00 ± 0.00
9,ResNet50,source_3class,"Three-way α=0.90, β=0.60",97.87 ± 1.26,1.39 ± 0.51,0.74 ± 0.81,99.05 ± 0.51,46.67 ± 32.93
12,ResNet50,source_3class,Two-way α=0.90,97.87 ± 1.26,0.00 ± 0.00,2.13 ± 1.26,99.05 ± 0.51,46.67 ± 32.93
14,ResNet50,zero_shot_target_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,87.47 ± 2.17,0.00 ± 0.00
16,ResNet50,zero_shot_target_3class,"Three-way α=0.90, β=0.60",90.53 ± 4.03,7.73 ± 2.97,1.73 ± 1.24,90.72 ± 1.30,31.54 ± 12.94
19,ResNet50,zero_shot_target_3class,Two-way α=0.90,90.53 ± 4.03,0.00 ± 0.00,9.47 ± 4.03,90.72 ± 1.30,31.54 ± 12.94


In [25]:
table_b = ablation_summary_df[
    ablation_summary_df["scheme"] == "Three-way"
].copy()

table_b["threshold_pair"] = table_b.apply(
    lambda row: f"α={row['alpha']:.2f}, β={row['beta']:.2f}",
    axis=1
)

table_b_formatted = pd.DataFrame({
    "Model": table_b["model"],
    "Dataset": table_b["dataset"],
    "Threshold Pair": table_b["threshold_pair"],
    "Accept (%)": table_b.apply(lambda row: format_mean_std(row, "accept_percentage"), axis=1),
    "Defer (%)": table_b.apply(lambda row: format_mean_std(row, "defer_percentage"), axis=1),
    "Reject (%)": table_b.apply(lambda row: format_mean_std(row, "reject_percentage"), axis=1),
    "Accept Accuracy (%)": table_b.apply(lambda row: format_mean_std(row, "accept_accuracy"), axis=1),
    "Wrong Capture Rate (%)": table_b.apply(lambda row: format_mean_std(row, "wrong_capture_rate"), axis=1)
})

table_b_formatted = table_b_formatted.sort_values(["Dataset", "Model", "Threshold Pair"])

threshold_results_path = "ablation_threshold_sensitivity_results.csv"
table_b_formatted.to_csv(threshold_results_path, index=False)

print("Saved:", threshold_results_path)
display(table_b_formatted)

Saved: ablation_threshold_sensitivity_results.csv


,Model,Dataset,Threshold Pair,Accept (%),Defer (%),Reject (%),Accept Accuracy (%),Wrong Capture Rate (%)
1,ResNet50,finetuned_target_3class,"α=0.85, β=0.50",96.67 ± 2.42,3.33 ± 2.42,0.00 ± 0.00,99.17 ± 0.50,62.67 ± 33.63
2,ResNet50,finetuned_target_3class,"α=0.90, β=0.60",96.27 ± 2.69,2.80 ± 2.36,0.93 ± 1.00,99.45 ± 0.51,70.00 ± 37.12
3,ResNet50,finetuned_target_3class,"α=0.95, β=0.70",95.73 ± 2.97,2.13 ± 0.98,2.13 ± 2.04,99.72 ± 0.34,83.33 ± 21.08
8,ResNet50,source_3class,"α=0.85, β=0.50",98.15 ± 1.40,1.85 ± 1.40,0.00 ± 0.00,98.96 ± 0.54,40.00 ± 37.95
9,ResNet50,source_3class,"α=0.90, β=0.60",97.87 ± 1.26,1.39 ± 0.51,0.74 ± 0.81,99.05 ± 0.51,46.67 ± 32.93
10,ResNet50,source_3class,"α=0.95, β=0.70",96.94 ± 1.77,2.13 ± 0.86,0.93 ± 0.97,99.24 ± 0.64,54.67 ± 39.42
15,ResNet50,zero_shot_target_3class,"α=0.85, β=0.50",92.93 ± 3.23,6.40 ± 2.85,0.67 ± 0.60,89.66 ± 1.34,22.32 ± 8.93
16,ResNet50,zero_shot_target_3class,"α=0.90, β=0.60",90.53 ± 4.03,7.73 ± 2.97,1.73 ± 1.24,90.72 ± 1.30,31.54 ± 12.94
17,ResNet50,zero_shot_target_3class,"α=0.95, β=0.70",87.60 ± 5.34,8.67 ± 3.07,3.73 ± 2.41,92.11 ± 1.63,43.19 ± 15.97


In [33]:
import numpy as np
import pandas as pd

def compute_extended_decision_metrics(
    df,
    scheme_name,
    alpha=None,
    beta=None
):
    n = len(df)

    if n == 0:
        raise ValueError("Cannot calculate metrics for an empty dataframe.")

    regions = df["decision_region"]
    correct = df["correct"].astype(bool)

    accept = regions == "Accept"
    defer = regions == "Defer"
    reject = regions == "Reject"

    def region_accuracy(mask):
        if mask.sum() == 0:
            return np.nan
        return 100.0 * correct[mask].mean()

    wrong = ~correct
    total_wrong = int(wrong.sum())

    wrong_captured = int((wrong & (defer | reject)).sum())

    wrong_capture_rate = (
        100.0 * wrong_captured / total_wrong
        if total_wrong > 0
        else np.nan
    )

    return {
        "scheme": scheme_name,
        "alpha": alpha,
        "beta": beta,
        "accept_pct": 100.0 * accept.mean(),
        "defer_pct": 100.0 * defer.mean(),
        "reject_pct": 100.0 * reject.mean(),
        "accept_accuracy": region_accuracy(accept),
        "defer_accuracy": region_accuracy(defer),
        "reject_accuracy": region_accuracy(reject),
        "wrong_capture_rate": wrong_capture_rate
    }

print("compute_extended_decision_metrics is ready.")

compute_extended_decision_metrics is ready.


In [34]:
two_way_alphas = [0.85, 0.90, 0.95]

three_way_thresholds = [
    (0.95, 0.70),
    (0.90, 0.60),
    (0.85, 0.50)
]

extended_rows = []

group_cols = ["model", "dataset", "seed"]

for (model_name, dataset_name, seed), group_df in all_predictions_df.groupby(group_cols):
    group_df = group_df.copy()

    # A. Standard hard prediction
    standard_df = group_df.copy()
    standard_df["decision_region"] = "Accept"

    result = compute_extended_decision_metrics(
        standard_df,
        scheme_name="Standard",
        alpha=None,
        beta=None
    )
    result["model"] = model_name
    result["dataset"] = dataset_name
    result["seed"] = seed
    extended_rows.append(result)

    # B. Two-way decision
    for alpha in two_way_alphas:
        two_way_df = group_df.copy()
        two_way_df["decision_region"] = np.where(
            two_way_df["confidence"] >= alpha,
            "Accept",
            "Reject"
        )

        result = compute_extended_decision_metrics(
            two_way_df,
            scheme_name="Two-way",
            alpha=alpha,
            beta=None
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        extended_rows.append(result)

    # C. Three-way decision
    for alpha, beta in three_way_thresholds:
        three_way_df = group_df.copy()

        conditions = [
            three_way_df["confidence"] >= alpha,
            (three_way_df["confidence"] >= beta) &
            (three_way_df["confidence"] < alpha),
            three_way_df["confidence"] < beta
        ]

        choices = ["Accept", "Defer", "Reject"]

        three_way_df["decision_region"] = np.select(
            conditions,
            choices,
            default="Reject"
        )

        result = compute_extended_decision_metrics(
            three_way_df,
            scheme_name="Three-way",
            alpha=alpha,
            beta=beta
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        extended_rows.append(result)


extended_ablation_df = pd.DataFrame(extended_rows)

extended_ablation_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_ablation_extended_region_metrics_per_seed.csv"
)

extended_ablation_df.to_csv(
    extended_ablation_path,
    index=False
)

print("Saved:", extended_ablation_path)

display(extended_ablation_df.head())

Saved: cross_dataset_transfer_20261027-050517/resnet50_ablation_extended_region_metrics_per_seed.csv


,scheme,alpha,beta,accept_pct,defer_pct,reject_pct,accept_accuracy,defer_accuracy,reject_accuracy,wrong_capture_rate,model,dataset,seed
0,Standard,NaN,NaN,100.000000,0.0,0.000000,96.666667,NaN,NaN,0.0,ResNet50,finetuned_target_3class,42
1,Two-way,0.85,NaN,92.666667,0.0,7.333333,99.280576,NaN,63.636364,80.0,ResNet50,finetuned_target_3class,42
2,Two-way,0.90,NaN,92.000000,0.0,8.000000,100.000000,NaN,58.333333,100.0,ResNet50,finetuned_target_3class,42
3,Two-way,0.95,NaN,90.666667,0.0,9.333333,100.000000,NaN,64.285714,100.0,ResNet50,finetuned_target_3class,42
4,Three-way,0.95,0.7,90.666667,4.0,5.333333,100.000000,83.333333,50.000000,100.0,ResNet50,finetuned_target_3class,42


In [35]:
print("Total rows:", len(extended_ablation_df))

print("\nRows by decision scheme:")
print(extended_ablation_df["scheme"].value_counts())

print("\nNaN counts:")
print(extended_ablation_df.isna().sum())

print("\nModels:")
print(extended_ablation_df["model"].unique())

print("\nSeeds:")
print(sorted(extended_ablation_df["seed"].unique()))

Total rows: 105

Rows by decision scheme:
scheme
Two-way      45
Three-way    45
Standard     15
Name: count, dtype: int64

NaN counts:
scheme                 0
alpha                 15
beta                  60
accept_pct             0
defer_pct              0
reject_pct             0
accept_accuracy        0
defer_accuracy        60
reject_accuracy       34
wrong_capture_rate     0
model                  0
dataset                0
seed                   0
dtype: int64

Models:
['ResNet50']

Seeds:
[np.int64(42), np.int64(123), np.int64(777), np.int64(2024), np.int64(3407)]


In [36]:
missing_reject = extended_ablation_df[
    extended_ablation_df["reject_accuracy"].isna()
]

assert (missing_reject["reject_pct"] == 0).all(), \
    "Unexpected NaN: Reject accuracy missing despite nonempty Reject region."

print("Verified: All missing Reject accuracies correspond to empty regions.")

Verified: All missing Reject accuracies correspond to empty regions.


In [37]:
main_table_df = extended_ablation_df[
    (
        (extended_ablation_df["scheme"] == "Standard") |
        (
            (extended_ablation_df["scheme"] == "Two-way") &
            (extended_ablation_df["alpha"] == 0.90)
        ) |
        (
            (extended_ablation_df["scheme"] == "Three-way") &
            (extended_ablation_df["alpha"] == 0.90) &
            (extended_ablation_df["beta"] == 0.60)
        )
    )
].copy()

In [38]:
print("Main comparison rows:", len(main_table_df))
print(main_table_df.groupby(["dataset", "scheme"]).size())

assert len(main_table_df) == 45, "Unexpected number of rows."

Main comparison rows: 45
dataset                  scheme   
finetuned_target_3class  Standard     5
                         Three-way    5
                         Two-way      5
source_3class            Standard     5
                         Three-way    5
                         Two-way      5
zero_shot_target_3class  Standard     5
                         Three-way    5
                         Two-way      5
dtype: int64


In [40]:
summary_metrics = [
    "accept_pct",
    "accept_accuracy",
    "defer_pct",
    "defer_accuracy",
    "reject_pct",
    "reject_accuracy",
    "wrong_capture_rate"
]

summary_rows = []

for keys, group in main_table_df.groupby(
    ["model", "dataset", "scheme", "alpha", "beta"],
    dropna=False
):
    model_name, dataset_name, scheme, alpha, beta = keys

    row = {
        "model": model_name,
        "dataset": dataset_name,
        "scheme": scheme,
        "alpha": alpha,
        "beta": beta
    }

    for metric in summary_metrics:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=0)

    summary_rows.append(row)

main_summary_df = pd.DataFrame(summary_rows)

print("Summary rows:", len(main_summary_df))
display(main_summary_df)

Summary rows: 9


,model,dataset,scheme,alpha,beta,accept_pct_mean,accept_pct_std,accept_accuracy_mean,accept_accuracy_std,defer_pct_mean,defer_pct_std,defer_accuracy_mean,defer_accuracy_std,reject_pct_mean,reject_pct_std,reject_accuracy_mean,reject_accuracy_std,wrong_capture_rate_mean,wrong_capture_rate_std
0,ResNet50,finetuned_target_3class,Standard,NaN,NaN,100.000000,0.000000,97.733333,1.236482,0.000000,0.000000,NaN,NaN,0.000000,0.000000,NaN,NaN,0.000000,0.000000
1,ResNet50,finetuned_target_3class,Three-way,0.9,0.6,96.266667,2.686592,99.452768,0.505166,2.800000,2.362673,57.727273,33.203604,0.933333,0.997775,25.000000,35.355339,70.000000,37.118429
2,ResNet50,finetuned_target_3class,Two-way,0.9,NaN,96.266667,2.686592,99.452768,0.505166,0.000000,0.000000,NaN,NaN,3.733333,2.686592,56.666667,33.082389,70.000000,37.118429
3,ResNet50,source_3class,Standard,NaN,NaN,100.000000,0.000000,98.333333,0.555556,0.000000,0.000000,NaN,NaN,0.000000,0.000000,NaN,NaN,0.000000,0.000000
4,ResNet50,source_3class,Three-way,0.9,0.6,97.870370,1.262795,99.054652,0.514452,1.388889,0.507151,69.333333,19.020457,0.740741,0.807204,40.000000,42.426407,46.666667,32.930904
5,ResNet50,source_3class,Two-way,0.9,NaN,97.870370,1.262795,99.054652,0.514452,0.000000,0.000000,NaN,NaN,2.129630,1.262795,64.000000,22.251092,46.666667,32.930904
6,ResNet50,zero_shot_target_3class,Standard,NaN,NaN,100.000000,0.000000,87.466667,2.166410,0.000000,0.000000,NaN,NaN,0.000000,0.000000,NaN,NaN,0.000000,0.000000
7,ResNet50,zero_shot_target_3class,Three-way,0.9,0.6,90.533333,4.030991,90.724506,1.300153,7.733333,2.969474,60.083333,21.708933,1.733333,1.236482,50.416667,29.589201,31.539683,12.937005
8,ResNet50,zero_shot_target_3class,Two-way,0.9,NaN,90.533333,4.030991,90.724506,1.300153,0.000000,0.000000,NaN,NaN,9.466667,4.030991,57.761905,14.042049,31.539683,12.937005


In [42]:
def fmt_mean_std(row, metric):
    mean_value = row[f"{metric}_mean"]
    std_value = row[f"{metric}_std"]

    if pd.isna(mean_value):
        return "-"

    if pd.isna(std_value):
        return f"{mean_value:.2f} ± N/A"

    return f"{mean_value:.2f} ± {std_value:.2f}"


def decision_label(row):
    if row["scheme"] == "Standard":
        return "Standard"
    elif row["scheme"] == "Two-way":
        return f"Two-way α={row['alpha']:.2f}"
    else:
        return (
            f"Three-way α={row['alpha']:.2f}, "
            f"β={row['beta']:.2f}"
        )


paper_extended_table = pd.DataFrame({
    "Model": main_summary_df["model"],
    "Dataset": main_summary_df["dataset"],
    "Decision": main_summary_df.apply(decision_label, axis=1),

    "Accept %": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "accept_pct"), axis=1
    ),
    "Accept Acc.": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "accept_accuracy"), axis=1
    ),
    "Defer %": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "defer_pct"), axis=1
    ),
    "Defer Acc.": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "defer_accuracy"), axis=1
    ),
    "Reject %": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "reject_pct"), axis=1
    ),
    "Reject Acc.": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "reject_accuracy"), axis=1
    ),
    "WCR": main_summary_df.apply(
        lambda row: fmt_mean_std(row, "wrong_capture_rate"), axis=1
    )
})

paper_extended_table = paper_extended_table.sort_values(
    ["Dataset", "Model", "Decision"]
).reset_index(drop=True)

paper_extended_table_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_ablation_main_paper_extended_region_table.csv"
)

paper_extended_table.to_csv(
    paper_extended_table_path,
    index=False
)

print("Saved:", paper_extended_table_path)
print("Total rows:", len(paper_extended_table))

display(paper_extended_table)

Saved: cross_dataset_transfer_20261027-050517/resnet50_ablation_main_paper_extended_region_table.csv
Total rows: 9


,Model,Dataset,Decision,Accept %,Accept Acc.,Defer %,Defer Acc.,Reject %,Reject Acc.,WCR
0,ResNet50,finetuned_target_3class,Standard,100.00 ± 0.00,97.73 ± 1.24,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00
1,ResNet50,finetuned_target_3class,"Three-way α=0.90, β=0.60",96.27 ± 2.69,99.45 ± 0.51,2.80 ± 2.36,57.73 ± 33.20,0.93 ± 1.00,25.00 ± 35.36,70.00 ± 37.12
2,ResNet50,finetuned_target_3class,Two-way α=0.90,96.27 ± 2.69,99.45 ± 0.51,0.00 ± 0.00,-,3.73 ± 2.69,56.67 ± 33.08,70.00 ± 37.12
3,ResNet50,source_3class,Standard,100.00 ± 0.00,98.33 ± 0.56,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00
4,ResNet50,source_3class,"Three-way α=0.90, β=0.60",97.87 ± 1.26,99.05 ± 0.51,1.39 ± 0.51,69.33 ± 19.02,0.74 ± 0.81,40.00 ± 42.43,46.67 ± 32.93
5,ResNet50,source_3class,Two-way α=0.90,97.87 ± 1.26,99.05 ± 0.51,0.00 ± 0.00,-,2.13 ± 1.26,64.00 ± 22.25,46.67 ± 32.93
6,ResNet50,zero_shot_target_3class,Standard,100.00 ± 0.00,87.47 ± 2.17,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00
7,ResNet50,zero_shot_target_3class,"Three-way α=0.90, β=0.60",90.53 ± 4.03,90.72 ± 1.30,7.73 ± 2.97,60.08 ± 21.71,1.73 ± 1.24,50.42 ± 29.59,31.54 ± 12.94
8,ResNet50,zero_shot_target_3class,Two-way α=0.90,90.53 ± 4.03,90.72 ± 1.30,0.00 ± 0.00,-,9.47 ± 4.03,57.76 ± 14.04,31.54 ± 12.94


In [43]:
gap_source_df = extended_ablation_df[
    (extended_ablation_df["scheme"] == "Three-way") &
    (extended_ablation_df["alpha"] == 0.90) &
    (extended_ablation_df["beta"] == 0.60)
].copy()

# Calculate paired accuracy gaps within each seed
gap_source_df["gap_ad"] = (
    gap_source_df["accept_accuracy"] -
    gap_source_df["defer_accuracy"]
)

gap_source_df["gap_dr"] = (
    gap_source_df["defer_accuracy"] -
    gap_source_df["reject_accuracy"]
)

gap_summary_rows = []

for (model_name, dataset_name), group in gap_source_df.groupby(
    ["model", "dataset"]
):
    row = {
        "Model": model_name,
        "Dataset": dataset_name
    }

    for metric, label in [
        ("accept_accuracy", "Accept Acc."),
        ("defer_accuracy", "Defer Acc."),
        ("reject_accuracy", "Reject Acc.")
    ]:
        row[f"{label} Mean"] = group[metric].mean()
        row[f"{label} Std"] = group[metric].std(ddof=0)
        row[f"{label} Valid Seeds"] = group[metric].count()

    row["Gap(A,D)"] = group["gap_ad"].mean()
    row["Gap(A,D) Std"] = group["gap_ad"].std(ddof=0)
    row["Gap(A,D) Valid Seeds"] = group["gap_ad"].count()

    row["Gap(D,R)"] = group["gap_dr"].mean()
    row["Gap(D,R) Std"] = group["gap_dr"].std(ddof=0)
    row["Gap(D,R) Valid Seeds"] = group["gap_dr"].count()

    gap_summary_rows.append(row)

gap_table_raw = pd.DataFrame(gap_summary_rows)

print("Gap summary rows:", len(gap_table_raw))
display(gap_table_raw)

Gap summary rows: 3


,Model,Dataset,Accept Acc. Mean,Accept Acc. Std,Accept Acc. Valid Seeds,Defer Acc. Mean,Defer Acc. Std,Defer Acc. Valid Seeds,Reject Acc. Mean,Reject Acc. Std,Reject Acc. Valid Seeds,"Gap(A,D)","Gap(A,D) Std","Gap(A,D) Valid Seeds","Gap(D,R)","Gap(D,R) Std","Gap(D,R) Valid Seeds"
0,ResNet50,finetuned_target_3class,99.452768,0.505166,5,57.727273,33.203604,5,25.000000,35.355339,3,41.725495,33.382212,5,21.212121,29.998470,3
1,ResNet50,source_3class,99.054652,0.514452,5,69.333333,19.020457,5,40.000000,42.426407,4,29.721318,19.113044,5,21.666667,44.628093,4
2,ResNet50,zero_shot_target_3class,90.724506,1.300153,5,60.083333,21.708933,5,50.416667,29.589201,4,30.641173,21.332050,5,14.687500,49.830768,4


In [44]:
def fmt_acc(mean_value, std_value, valid_seeds):
    if pd.isna(mean_value):
        return "-"

    if pd.isna(std_value):
        return f"{mean_value:.2f} (n={valid_seeds})"

    return f"{mean_value:.2f} ± {std_value:.2f} (n={valid_seeds})"


def fmt_gap(mean_value, std_value, valid_seeds):
    if pd.isna(mean_value):
        return "-"

    if pd.isna(std_value):
        return f"{mean_value:.2f} (n={valid_seeds})"

    return f"{mean_value:.2f} ± {std_value:.2f} (n={valid_seeds})"


gap_table = pd.DataFrame({
    "Model": gap_table_raw["Model"],
    "Dataset": gap_table_raw["Dataset"],

    "Accept Acc.": gap_table_raw.apply(
        lambda row: fmt_acc(
            row["Accept Acc. Mean"],
            row["Accept Acc. Std"],
            int(row["Accept Acc. Valid Seeds"])
        ),
        axis=1
    ),

    "Defer Acc.": gap_table_raw.apply(
        lambda row: fmt_acc(
            row["Defer Acc. Mean"],
            row["Defer Acc. Std"],
            int(row["Defer Acc. Valid Seeds"])
        ),
        axis=1
    ),

    "Reject Acc.": gap_table_raw.apply(
        lambda row: fmt_acc(
            row["Reject Acc. Mean"],
            row["Reject Acc. Std"],
            int(row["Reject Acc. Valid Seeds"])
        ),
        axis=1
    ),

    "Gap(A,D)": gap_table_raw.apply(
        lambda row: fmt_gap(
            row["Gap(A,D)"],
            row["Gap(A,D) Std"],
            int(row["Gap(A,D) Valid Seeds"])
        ),
        axis=1
    ),

    "Gap(D,R)": gap_table_raw.apply(
        lambda row: fmt_gap(
            row["Gap(D,R)"],
            row["Gap(D,R) Std"],
            int(row["Gap(D,R) Valid Seeds"])
        ),
        axis=1
    )
})

gap_table = gap_table.sort_values(
    ["Dataset", "Model"]
).reset_index(drop=True)

gap_table_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_ablation_three_way_region_accuracy_gap_table.csv"
)

gap_table.to_csv(gap_table_path, index=False)

print("Saved:", gap_table_path)
print("Total rows:", len(gap_table))

display(gap_table)

Saved: cross_dataset_transfer_20261027-050517/resnet50_ablation_three_way_region_accuracy_gap_table.csv
Total rows: 3


,Model,Dataset,Accept Acc.,Defer Acc.,Reject Acc.,"Gap(A,D)","Gap(D,R)"
0,ResNet50,finetuned_target_3class,99.45 ± 0.51 (n=5),57.73 ± 33.20 (n=5),25.00 ± 35.36 (n=3),41.73 ± 33.38 (n=5),21.21 ± 30.00 (n=3)
1,ResNet50,source_3class,99.05 ± 0.51 (n=5),69.33 ± 19.02 (n=5),40.00 ± 42.43 (n=4),29.72 ± 19.11 (n=5),21.67 ± 44.63 (n=4)
2,ResNet50,zero_shot_target_3class,90.72 ± 1.30 (n=5),60.08 ± 21.71 (n=5),50.42 ± 29.59 (n=4),30.64 ± 21.33 (n=5),14.69 ± 49.83 (n=4)


In [45]:
for _, row in gap_table_raw.sort_values(["Dataset", "Model"]).iterrows():
    print(
        f"{row['Model']} | {row['Dataset']}: "
        f"Gap(A,D) = {row['Accept Acc. Mean']:.2f} - {row['Defer Acc. Mean']:.2f} = {row['Gap(A,D)']:.2f}; "
        f"Gap(D,R) = {row['Defer Acc. Mean']:.2f} - {row['Reject Acc. Mean']:.2f} = {row['Gap(D,R)']:.2f}"
    )

ResNet50 | finetuned_target_3class: Gap(A,D) = 99.45 - 57.73 = 41.73; Gap(D,R) = 57.73 - 25.00 = 21.21
ResNet50 | source_3class: Gap(A,D) = 99.05 - 69.33 = 29.72; Gap(D,R) = 69.33 - 40.00 = 21.67
ResNet50 | zero_shot_target_3class: Gap(A,D) = 90.72 - 60.08 = 30.64; Gap(D,R) = 60.08 - 50.42 = 14.69


In [49]:
import os
import zipfile

FINAL_EXPORT_NAME = "ResNet50V2_CrossDataset_Ablation_All_Outputs"

BASE_DIR = OUTPUT_DIR
ZIP_PATH = os.path.join(
    "/kaggle/working",
    f"{FINAL_EXPORT_NAME}.zip"
)

items_to_export = []

# 1. ResNet-50 trained models and histories
for folder in ["models", "histories", "history", "prediction_csvs"]:
    folder_path = os.path.join(BASE_DIR, folder)

    if os.path.isdir(folder_path):
        for root, dirs, files in os.walk(folder_path):
            for filename in files:
                file_path = os.path.join(root, filename)

                if folder == "models" and not filename.startswith("ResNet50_"):
                    continue

                if folder in ["histories", "history"] and not filename.startswith("ResNet50_"):
                    continue

                if folder == "prediction_csvs" and not filename.startswith("test_predictions_ResNet50_"):
                    continue

                items_to_export.append(file_path)

# 2. ResNet-50 experiment CSVs and result files
if os.path.isdir(BASE_DIR):
    for filename in os.listdir(BASE_DIR):
        file_path = os.path.join(BASE_DIR, filename)

        if (
            os.path.isfile(file_path)
            and filename.lower().startswith("resnet50_")
            and filename.lower().endswith(
                (".csv", ".json", ".pkl", ".png")
            )
        ):
            items_to_export.append(file_path)

# 3. Remove duplicates
items_to_export = sorted(set(items_to_export))

print(f"Files to export: {len(items_to_export)}")

for file_path in items_to_export:
    print(os.path.relpath(file_path, BASE_DIR))

if not items_to_export:
    raise FileNotFoundError(
        "No ResNet-50 experiment outputs found. "
        "Check OUTPUT_DIR and run the experiment cells first."
    )

# 4. Create final ZIP
with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zip_file:

    for file_path in items_to_export:
        relative_path = os.path.relpath(file_path, BASE_DIR)

        zip_file.write(
            file_path,
            arcname=relative_path
        )

# 5. Verify archive
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    corrupted_file = zip_file.testzip()

    if corrupted_file is not None:
        raise RuntimeError(
            f"ZIP verification failed: {corrupted_file}"
        )

print("\nFinal export created successfully:")
print(ZIP_PATH)

print(
    "ZIP size:",
    round(os.path.getsize(ZIP_PATH) / (1024 ** 2), 2),
    "MB"
)

Files to export: 45
histories/ResNet50_seed_123_source_history.pkl
histories/ResNet50_seed_123_target_finetune_history.pkl
histories/ResNet50_seed_2024_source_history.pkl
histories/ResNet50_seed_2024_target_finetune_history.pkl
histories/ResNet50_seed_3407_source_history.pkl
histories/ResNet50_seed_3407_target_finetune_history.pkl
histories/ResNet50_seed_42_source_history.pkl
histories/ResNet50_seed_42_target_finetune_history.pkl
histories/ResNet50_seed_777_source_history.pkl
histories/ResNet50_seed_777_target_finetune_history.pkl
models/ResNet50_seed_123_source.keras
models/ResNet50_seed_123_target_finetuned.keras
models/ResNet50_seed_2024_source.keras
models/ResNet50_seed_2024_target_finetuned.keras
models/ResNet50_seed_3407_source.keras
models/ResNet50_seed_3407_target_finetuned.keras
models/ResNet50_seed_42_source.keras
models/ResNet50_seed_42_target_finetuned.keras
models/ResNet50_seed_777_source.keras
models/ResNet50_seed_777_target_finetuned.keras
prediction_csvs/test_prediction

In [50]:
from IPython.display import display, FileLink, HTML
import os

zip_path = ZIP_PATH

if not os.path.isfile(zip_path):
    raise FileNotFoundError(f"ZIP not found: {zip_path}")

size_mb = os.path.getsize(zip_path) / (1024 ** 2)

print(f"ZIP ready: {os.path.basename(zip_path)}")
print(f"Size: {size_mb:.2f} MB")

display(FileLink(zip_path))

display(HTML(
    "<p>Click the file link above to download your complete experiment archive.</p>"
))

ZIP ready: ResNet50V2_CrossDataset_Ablation_All_Outputs.zip
Size: 1475.97 MB


/kaggle/working/ResNet50V2_CrossDataset_Ablation_All_Outputs.zip

In [51]:
import os
from IPython.display import display, HTML

filename = os.path.basename(ZIP_PATH)

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)

display(HTML(f"""
<a href="/files/{filename}"
   download="{filename}"
   target="_blank"
   style="
       display:inline-block;
       background:#1976D2;
       color:white;
       padding:14px 24px;
       border-radius:8px;
       font-size:16px;
       font-weight:bold;
       text-decoration:none;
   ">
   ⬇ Download ResNet-50 ZIP
</a>
"""))

print(f"File: {filename}")
print(f"Size: {os.path.getsize(ZIP_PATH) / 1024**2:.2f} MB")

File: ResNet50V2_CrossDataset_Ablation_All_Outputs.zip
Size: 1475.97 MB


In [47]:
import os
import zipfile

zip_path = os.path.join(
    OUTPUT_DIR,
    "resnet50_ablation_all_analysis_outputs.zip"
)

files_to_zip = [
    "resnet50_ablation_decision_mechanism_results.csv",
    "resnet50_ablation_decision_mechanism_summary_mean_std.csv",
    "resnet50_ablation_table_A_standard_two_way_three_way.csv",
    "resnet50_ablation_threshold_sensitivity_results.csv",
    "resnet50_ablation_extended_region_metrics_per_seed.csv",
    "resnet50_ablation_main_paper_extended_region_table.csv",
    "resnet50_ablation_three_way_region_accuracy_gap_table.csv"
]

missing_files = [
    filename for filename in files_to_zip
    if not os.path.isfile(os.path.join(OUTPUT_DIR, filename))
]

if missing_files:
    raise FileNotFoundError(
        "Missing files:\n" + "\n".join(missing_files)
    )

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in files_to_zip:
        zipf.write(
            os.path.join(OUTPUT_DIR, filename),
            arcname=filename
        )

with zipfile.ZipFile(zip_path, "r") as zipf:
    assert zipf.testzip() is None

print("Created:", zip_path)
print("Files included:", len(files_to_zip))

from IPython.display import FileLink, display
display(FileLink(zip_path))

Created: cross_dataset_transfer_20261027-050517/resnet50_ablation_all_analysis_outputs.zip
Files included: 7


/kaggle/working/cross_dataset_transfer_20261027-050517/resnet50_ablation_all_analysis_outputs.zip

In [48]:
import os
from IPython.display import display, Javascript, FileLink

zip_path = ZIP_PATH

if not os.path.isfile(zip_path):
    raise FileNotFoundError(f"ZIP not found: {zip_path}")

print("ZIP ready:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024**2, 2), "MB")

# Attempt automatic browser download
display(Javascript("""
(async function() {
    const filename = "ResNet50_CrossDataset_Ablation_All_Outputs.zip";
    const url = "/files/" + encodeURIComponent(filename);

    const link = document.createElement("a");
    link.href = url;
    link.download = filename;
    document.body.appendChild(link);
    link.click();
    link.remove();

    console.log("Download requested:", filename);
})();
"""))

# Manual fallback link
display(FileLink(zip_path))

ZIP ready: /kaggle/working/ResNet50_CrossDataset_Ablation_All_Outputs.zip
Size: 1475.97 MB


<IPython.core.display.Javascript object>

/kaggle/working/ResNet50_CrossDataset_Ablation_All_Outputs.zip

In [ ]:
SOURCE_DATASET_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DATASET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"